# Quantalytics — Phase 2: Bronze → Silver (IBM + NOAA)

**Start here.** This notebook reads two **existing** Bronze Delta tables and builds two simple Silver Delta tables. It also creates an audit table and a quarantine table.

Run cells **from top to bottom**. It will **not** delete or overwrite your friend's Bronze tables. Each cell has comments explaining its purpose. Change widget parameters to reprocess a particular source file or historical date range.

**Prerequisite:** your friend must have run both Bronze loaders in the *same Databricks workspace*. If the Bronze tables are empty, fix that first. Bronze `load_timestamp` and Bronze audit logs also require the corrections in `Bronze_Fixes_For_Friend.md`.

## Cell 1 — Imports, settings, and parameters

In [0]:
import uuid
import json
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType, MapType
from pyspark.sql.window import Window
from delta.tables import DeltaTable

spark.conf.set("spark.sql.session.timeZone", "UTC")
CATALOG = "workspace"     # This matches your friend's Bronze notebooks.
IBM_BRONZE = f"{CATALOG}.bronze.ibm_calibration_raw"
NOAA_BRONZE = f"{CATALOG}.bronze.noaa_space_weather_raw"
IBM_SILVER = f"{CATALOG}.silver.ibm_qubit_calibration"
NOAA_SILVER = f"{CATALOG}.silver.noaa_environment"
LOG_TABLE = f"{CATALOG}.silver.pipeline_execution_logs"
BAD_TABLE = f"{CATALOG}.silver.quarantined_records"

# Widgets appear above the Databricks notebook. Blank filters mean all data.
# Keep widget values on a second run instead of recreating them.
def make_widget(name, kind, default, choices=None):
    try:
        dbutils.widgets.get(name)
    except Exception:
        if kind == "dropdown":
            dbutils.widgets.dropdown(name, default, choices)
        else:
            dbutils.widgets.text(name, default)

make_widget("dataset", "dropdown", "both", ["both", "ibm", "noaa"])
make_widget("mode", "dropdown", "full", ["full", "incremental", "backfill"])
make_widget("source_file", "text", "")
make_widget("start_date", "text", "")            # e.g. 2026-09-01
make_widget("end_date", "text", "")              # e.g. 2026-09-30

DATASET = dbutils.widgets.get("dataset").lower()
MODE = dbutils.widgets.get("mode").lower()
SOURCE_FILE = dbutils.widgets.get("source_file").strip()
START_DATE = dbutils.widgets.get("start_date").strip()
END_DATE = dbutils.widgets.get("end_date").strip()

if MODE == "incremental" and not SOURCE_FILE:
    raise ValueError("For incremental mode, enter the new file's name in source_file.")
if MODE == "backfill" and not (START_DATE or END_DATE):
    raise ValueError("For backfill mode, enter start_date and/or end_date.")
for value in [START_DATE, END_DATE]:
    if value:
        datetime.strptime(value, "%Y-%m-%d")       # Check date format.

RUN_ID = str(uuid.uuid4())
RUN_START = datetime.now(timezone.utc).replace(tzinfo=None)
print("Run:", RUN_ID)
print("Parameters:", DATASET, MODE, SOURCE_FILE or "all files", START_DATE, END_DATE)

Run: bcc8b5ec-26d2-4698-b5b8-1205798e552d
Parameters: ibm incremental ibm_incremental_20261009T124928Z.csv  


## Cell 2 — Check the shared Bronze tables; create Silver operational tables

In [0]:
# Check access BEFORE creating any Silver data.
for table in ([IBM_BRONZE] if DATASET == "ibm" else
              [NOAA_BRONZE] if DATASET == "noaa" else [IBM_BRONZE, NOAA_BRONZE]):
    if not spark.catalog.tableExists(table):
        raise ValueError(f"Cannot find {table}. Ask your friend to run/share his Bronze notebook.")
    print(table, "rows:", spark.table(table).count())

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")

# Explicit schema for the AUDIT table (no inference).
LOG_SCHEMA = StructType([
    StructField("run_id", StringType(), False),
    StructField("stage", StringType(), False),
    StructField("dataset", StringType(), False),
    StructField("mode", StringType(), False),
    StructField("parameters", StringType(), True),
    StructField("started_at", TimestampType(), False),
    StructField("finished_at", TimestampType(), False),
    StructField("status", StringType(), False),
    StructField("input_rows", LongType(), False),
    StructField("rows_inserted", LongType(), False),
    StructField("rows_updated", LongType(), False),
    StructField("rows_quarantined", LongType(), False),
    StructField("error_message", StringType(), True),
    StructField("load_timestamp", TimestampType(), False),
])

# Same for rejected/changed-schema records.
BAD_SCHEMA = StructType([
    StructField("record_key", StringType(), True),
    StructField("dataset", StringType(), True),
    StructField("source_file", StringType(), True),
    StructField("raw_payload", StringType(), True),
    StructField("error_reason", StringType(), True),
    StructField("load_timestamp", TimestampType(), True),
])

spark.createDataFrame([], LOG_SCHEMA).write.format("delta").mode("ignore").saveAsTable(LOG_TABLE)
spark.createDataFrame([], BAD_SCHEMA).write.format("delta").mode("ignore").saveAsTable(BAD_TABLE)
print("Operational tables ready.")

workspace.bronze.ibm_calibration_raw rows: 456612
Operational tables ready.


## Cell 3 — Small reusable functions: filters, MERGE, quarantine, logging

In [0]:
def select_input(table, time_col):
    """Read Bronze, then optionally select one file or historical date range."""
    df = spark.table(table)
    if SOURCE_FILE:
        df = df.filter(F.col("source_file") == SOURCE_FILE)
    event_time = F.expr(f"try_cast({time_col} AS timestamp)")
    if START_DATE:
        df = df.filter(F.to_date(event_time) >= F.lit(START_DATE))
    if END_DATE:
        df = df.filter(F.to_date(event_time) <= F.lit(END_DATE))
    return df


def quarantine(bad_rows, dataset):
    """Keep faulty records for inspection; MERGE avoids duplicate bad rows."""
    rejected = (bad_rows.select("source_file", "raw_payload", "error_reason")
        .withColumn("dataset", F.lit(dataset))
        .withColumn("record_key", F.sha2(F.concat_ws("||", "dataset", "source_file", "raw_payload", "error_reason"), 256))
        .withColumn("load_timestamp", F.current_timestamp())
        .select("record_key", "dataset", "source_file", "raw_payload", "error_reason", "load_timestamp")
        .dropDuplicates(["record_key"]))
    n = rejected.count()
    if n:
        (DeltaTable.forName(spark, BAD_TABLE).alias("t")
         .merge(rejected.alias("s"), "t.record_key = s.record_key")
         .whenNotMatchedInsertAll()
         .execute())
    return n


def merge_silver(rows, table):
    """Create target if needed; update ONLY if actual source payload changed."""
    if not spark.catalog.tableExists(table):
        rows.limit(0).write.format("delta").saveAsTable(table)
    changes = {column: f"s.`{column}`" for column in rows.columns}
    (DeltaTable.forName(spark, table).alias("t")
     .merge(rows.alias("s"), "t.record_key = s.record_key")
     .whenMatchedUpdate(condition="t.source_payload_hash <> s.source_payload_hash", set=changes)
     .whenNotMatchedInsertAll()
     .execute())
    # Databricks reports how many rows the last MERGE inserted / updated.
    history = spark.sql(f"DESCRIBE HISTORY {table} LIMIT 1").first()
    metrics = (history["operationMetrics"] or {}) if history else {}
    return int(metrics.get("numTargetRowsInserted", 0)), int(metrics.get("numTargetRowsUpdated", 0))


def audit(dataset, started, status, total=0, inserted=0, updated=0, bad=0, error=None):
    """Every attempted Bronze-to-Silver table run receives one log row."""
    now = datetime.now(timezone.utc).replace(tzinfo=None)
    parameters = f"source_file={SOURCE_FILE or 'ALL'}; start_date={START_DATE or 'ALL'}; end_date={END_DATE or 'ALL'}"
    row = [(RUN_ID, "Bronze-to-Silver", dataset, MODE, parameters, started, now,
            status, int(total), int(inserted), int(updated), int(bad), error, now)]
    spark.createDataFrame(row, LOG_SCHEMA).write.format("delta").mode("append").saveAsTable(LOG_TABLE)

## Cell 4 — IBM: explicit raw JSON schema → clean qubit measurements → Silver

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, MapType# These EXACT names come from the IBM CSV your friend used for Bronze.

IBM_COLUMNS = [
    "Qubit", "T1 (us)", "T2 (us)", "Readout assignment error", "Init error",
    "Prob meas0 prep1", "Prob meas1 prep0", "Readout length (ns)",
    "ID error", "Single-qubit gate length (ns)", "RX error",
    "Z-axis rotation (rz) error", "√x (sx) error", "Pauli-X error",
    "XSLOW error", "CZ error", "Gate length (ns)", "RZZ error",
    "MEASURE error", "MEASURE_2 error", "Operational", "Backend_ID",
    "Calibration_Timestamp"
]
IBM_PAYLOAD_SCHEMA = StructType([StructField(c, StringType(), True) for c in IBM_COLUMNS])

if DATASET in ("both", "ibm"):
    started = datetime.now(timezone.utc).replace(tzinfo=None)
    total = bad_n = 0
    try:
        bronze = select_input(IBM_BRONZE, "raw_calibration_timestamp")
        total = bronze.count()

        # from_json imposes the explicit StructType. A MapType separately spots extra fields.
        df = (bronze
          .withColumn("p", F.from_json("raw_payload", IBM_PAYLOAD_SCHEMA))
          .withColumn("all_keys", F.map_keys(F.from_json("raw_payload", MapType(StringType(), StringType()))))
          .withColumn("extra_keys", F.array_except("all_keys", F.array(*[F.lit(c) for c in IBM_COLUMNS])))
          .select(
            "raw_payload", "source_file", "collection_timestamp", "extra_keys",
            F.trim(F.col("p.Backend_ID")).alias("backend_id"),
            F.col("p.Qubit").alias("qubit_text"),
            F.col("p.Calibration_Timestamp").alias("time_text"),
            F.col("p.`T1 (us)`").alias("t1_text"),
            F.col("p.`T2 (us)`").alias("t2_text"),
            F.col("p.`Readout assignment error`").alias("readout_text"),
            F.col("p.`Init error`").alias("init_text"),
            F.col("p.Operational").alias("operational_text")))

        df = (df
          .withColumn("qubit_id", F.expr("try_cast(qubit_text AS INT)"))
          .withColumn("calibration_timestamp", F.expr("try_cast(time_text AS TIMESTAMP)"))
          .withColumn("t1_us", F.expr("try_cast(t1_text AS DOUBLE)"))
          .withColumn("t2_us", F.expr("try_cast(t2_text AS DOUBLE)"))
          .withColumn("readout_error", F.expr("try_cast(readout_text AS DOUBLE)"))
          .withColumn("init_error", F.expr("try_cast(init_text AS DOUBLE)"))
          .withColumn("operational", F.when(F.lower(F.trim("operational_text")).isin("yes", "y", "true", "1"), True)
                                       .when(F.lower(F.trim("operational_text")).isin("no", "n", "false", "0"), False)))

        # A null source measurement is allowed, but a non-empty invalid number is not.
        problems = [
            F.when(F.col("backend_id").isNull() | (F.col("backend_id") == ""), "MISSING_BACKEND"),
            F.when(F.col("qubit_id").isNull() | (F.col("qubit_id") < 0), "INVALID_QUBIT"),
            F.when(F.col("calibration_timestamp").isNull(), "INVALID_TIMESTAMP"),
            F.when(F.size("extra_keys") > 0, "SCHEMA_DRIFT_NEW_COLUMN"),
            F.when(F.col("raw_payload").isNull(), "NULL_PAYLOAD"),
        ]
        for text_col, numeric_col in [("t1_text", "t1_us"), ("t2_text", "t2_us"),
                                      ("readout_text", "readout_error"), ("init_text", "init_error")]:
            problems.append(F.when((F.trim(F.col(text_col)) != "") & F.col(text_col).isNotNull()
                                   & F.col(numeric_col).isNull(), f"INVALID_{numeric_col.upper()}"))
        problems += [
            F.when(F.col("t1_us").isNotNull() & (F.col("t1_us") <= 0), "T1_NOT_POSITIVE"),
            F.when(F.col("t2_us").isNotNull() & (F.col("t2_us") <= 0), "T2_NOT_POSITIVE"),
            F.when(F.col("readout_error").isNotNull() & ~F.col("readout_error").between(0, 1), "READOUT_OUT_OF_RANGE"),
            F.when(F.col("init_error").isNotNull() & ~F.col("init_error").between(0, 1), "INIT_OUT_OF_RANGE"),
            F.when(F.col("operational_text").isNotNull() & (F.trim("operational_text") != "")
                   & F.col("operational").isNull(), "INVALID_OPERATIONAL"),
        ]
        df = df.withColumn("error_reason", F.concat_ws("; ", *problems))
        bad_n = quarantine(df.filter(F.col("error_reason") != ""), "ibm")
        good = df.filter(F.col("error_reason") == "")

        # Key = one qubit, in one processor, at one calibration timestamp.
        ready = (good
          .withColumn("record_key", F.sha2(F.concat_ws("||", "backend_id",
                           F.col("calibration_timestamp").cast("string"),
                           F.col("qubit_id").cast("string")), 256))
          .withColumn("source_payload_hash", F.sha2("raw_payload", 256))
          .withColumn("load_timestamp", F.current_timestamp())
          .select("record_key", "backend_id", "calibration_timestamp", "qubit_id",
                  "t1_us", "t2_us", "readout_error", "init_error", "operational",
                  "source_payload_hash", "source_file", "load_timestamp"))
        # A source file might have repeated keys: take one candidate per key.
        w = Window.partitionBy("record_key").orderBy(F.col("source_file").desc(),
                                                      F.col("source_payload_hash").desc())
        ready = ready.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")
        inserted, updated = merge_silver(ready, IBM_SILVER)
        audit("ibm", started, "SUCCESS", total, inserted, updated, bad_n)
        print(f"IBM SUCCESS: read {total}, inserted {inserted}, updated {updated}, quarantined {bad_n}")
        display(spark.table(IBM_SILVER).limit(10))
    except Exception as e:
        audit("ibm", started, "FAILURE", total, bad=bad_n, error=str(e)[:1500])
        raise

IBM SUCCESS: read 33228, inserted 0, updated 0, quarantined 0


record_key,backend_id,calibration_timestamp,qubit_id,t1_us,t2_us,readout_error,init_error,operational,source_payload_hash,source_file,load_timestamp
0000aa1e6a9b4954b8bd7a465752506806540d7f63b3e651a4f908ff6e3853bb,ibm_kingston,2026-08-29T23:45:11.000Z,118,250.66319539936435,148.13815514049176,0.0111083984375,null,true,f33da91f968e50f9bf0612b199a426ae34c0b83419c804d4ffc6215668c70051,ibm_full_20261009T114215Z.csv,2026-10-10T08:34:51.957Z
00022af9497702ec2c51f5ddffd7393fb90f5028d9d40e281eb1401606e619cd,ibm_fez,2026-04-19T09:25:40.000Z,96,171.98769938565448,160.7396700242285,0.0130615234375,null,true,ebe39dd2a5d6a398477c94e60de39e2a33c64b8ed2ce9e087b4cde334bac013f,ibm_full_20261009T114215Z.csv,2026-10-10T08:34:51.957Z
000445a0af12b646b26eb81bd45a13fa62a9f5f548cef408ad9f247ca93ec340,ibm_kingston,2026-04-01T03:34:28.000Z,148,276.05680666047687,209.93162496410596,0.0064697265625,null,true,360249c3a8b360766c277adecae5e55969595a801c20b46a52b1c94d9248fc85,ibm_full_20261009T114215Z.csv,2026-10-10T08:34:51.957Z
0004590f17dd6e500d54227961675571c1fac6fdadf4dfc16e58ec5428fbf1b8,ibm_marrakesh,2026-09-17T18:23:00.000Z,108,21.046321524547988,20.45534784989052,0.0078125,null,true,c87db94c25643f15395e7b96ee74a9ec1f5a0ebe0949edcbcea7b3e04335c61c,ibm_full_20261009T114215Z.csv,2026-10-10T08:34:51.957Z
000534190c60e5951a14c58a6f7c0146bca9f230d59074ea25e4b6711a07537d,ibm_fez,2026-05-24T23:17:13.000Z,108,98.78916270848894,100.27141139404002,0.010986328125,null,true,7900890ebd9f51be985838e924e33b0d116b7ab4016d14ce3fa38097da3654b8,ibm_full_20261009T114215Z.csv,2026-10-10T08:34:51.957Z
000550173262f785b9f80bdf9cc9bd9c76da3728cde4a0e2ccdf482e4daf0d81,ibm_marrakesh,2026-04-08T13:45:17.000Z,96,109.80191798898855,42.06003174273765,0.0252685546875,null,true,84ae30708a338263e4e4128a2bf46628edfc3f0beeee8fbfbe87f3ddada7e731,ibm_full_20261009T114215Z.csv,2026-10-10T08:34:51.957Z
0006f9f7b5837a0ec6c0a76c4d777130ed1ff6203bf0f67144675cdd4d14ea07,ibm_kingston,2026-05-06T09:31:34.000Z,81,352.1873826928176,349.77777019247947,0.00830078125,null,true,8aee3c38def84afbe0f03d0597f1e864e9f733064be130220f6bcc8ddae09e67,ibm_full_20261009T114215Z.csv,2026-10-10T08:34:51.957Z
0007d9ce6df546c129119b56d4ba1aa01cbb2cb5d2e47cd3b0022d1aaefe566c,ibm_marrakesh,2026-07-07T09:23:16.000Z,6,299.4684247157193,198.74463583839932,0.00244140625,null,true,f1f3d70fee57903867e1d128bdad210e242c532861adf595ee98be9fa47df73a,ibm_full_20261009T114215Z.csv,2026-10-10T08:34:51.957Z
000a1c6524613d4544457803853797e76a9eadbcf4f9c9e94ed7de49d546d2c3,ibm_fez,2026-07-24T18:08:20.000Z,76,160.00893209904797,91.10386834132545,0.0235595703125,null,true,b954827de8f65662d13e4f0170255825db5e8cf289a1e81ea9977be7eaf93dd8,ibm_full_20261009T114215Z.csv,2026-10-10T08:34:51.957Z
000d07bd8d3de0a87e61236e56125dc5ae347c0c6e946ec2877cf28dd01f3bf0,ibm_fez,2026-04-20T14:09:23.000Z,28,151.3425642351663,160.6456898651952,0.0078125,null,true,6e5710dcfa1fa60c5161b136c7100dadc38b04ea8a8108a75423c3c90d1a27fc,ibm_full_20261009T114215Z.csv,2026-10-10T08:34:51.957Z


## Cell 5 — NOAA: parse Kp and solar flux from the nested JSON → Silver

In [0]:
# The NOAA collector writes a JSON string with "source_record" and "parsed_fields".
# Historical data stores values in parsed_fields; live data stores them in source_record.
@F.udf(StringType())
def noaa_structure_error(raw):
    """Reject malformed JSON and unexpected top-level keys (schema drift)."""
    try:
        item = json.loads(raw)
        if not isinstance(item, dict):
            return "NOT_A_JSON_OBJECT"
        if set(item) != {"source_record", "parsed_fields"}:
            return "SCHEMA_DRIFT_NOAA_KEYS"
        if not isinstance(item["source_record"], (dict, str)):
            return "SCHEMA_DRIFT_SOURCE_RECORD_TYPE"
        if not isinstance(item["parsed_fields"], dict):
            return "SCHEMA_DRIFT_PARSED_FIELDS_TYPE"
        return None
    except Exception:
        return "INVALID_JSON"

if DATASET in ("both", "noaa"):
    started = datetime.now(timezone.utc).replace(tzinfo=None)
    total = bad_n = 0
    try:
        bronze = select_input(NOAA_BRONZE, "observation_timestamp")
        total = bronze.count()
        df = (bronze
          .withColumn("structure_error", noaa_structure_error("raw_payload"))
          .withColumn("observation_time", F.expr("try_cast(observation_timestamp AS TIMESTAMP)"))
          .withColumn("kp_text", F.coalesce(
               F.get_json_object("raw_payload", "$.source_record.Kp"),
               F.get_json_object("raw_payload", "$.source_record.kp_index"),
               F.get_json_object("raw_payload", "$.parsed_fields.estimated_planetary_k_index")))
          .withColumn("flux_text", F.coalesce(
               F.get_json_object("raw_payload", "$.source_record.flux"),
               F.get_json_object("raw_payload", "$.source_record.f10_7_radio_flux"),
               F.get_json_object("raw_payload", "$.parsed_fields.f10_7_radio_flux")))
          .withColumn("kp_index", F.expr("try_cast(kp_text AS DOUBLE)"))
          .withColumn("solar_flux", F.expr("try_cast(flux_text AS DOUBLE)"))
          .withColumn("observation_type", F.when(F.col("source") == "noaa_swpc_kp", "planetary_kp")
                                           .when(F.col("source") == "noaa_swpc_f107", "f10_7_flux"))
          .withColumn("source_unit", F.when(F.col("source") == "noaa_swpc_kp", "index")
                                     .when(F.col("source") == "noaa_swpc_f107", "sfu")))

        errors = [
          F.when(F.col("structure_error").isNotNull(), F.col("structure_error")),
          F.when(F.col("observation_time").isNull(), "INVALID_TIMESTAMP"),
          F.when(F.col("observation_type").isNull(), "UNKNOWN_SOURCE"),
          F.when(F.col("source_file").isNull(), "MISSING_SOURCE_FILE"),
          F.when((F.col("source") == "noaa_swpc_kp") &
                 (F.col("kp_index").isNull() | ~F.col("kp_index").between(0, 9)), "INVALID_KP"),
          F.when((F.col("source") == "noaa_swpc_f107") &
                 (F.col("solar_flux").isNull() | (F.col("solar_flux") <= 0)), "INVALID_SOLAR_FLUX"),
        ]
        df = df.withColumn("error_reason", F.concat_ws("; ", *errors))
        bad_n = quarantine(df.filter(F.col("error_reason") != ""), "noaa")
        good = df.filter(F.col("error_reason") == "")
        ready = (good
          .withColumn("record_key", F.sha2(F.concat_ws("||", "source", "observation_type",
                         F.col("observation_time").cast("string")), 256))
          .withColumn("source_payload_hash", F.sha2("raw_payload", 256))
          .withColumn("load_timestamp", F.current_timestamp())
          .select("record_key", "source", "observation_type",
                  F.col("observation_time").alias("observation_timestamp"),
                  "kp_index", "solar_flux", "source_unit", "source_payload_hash",
                  "source_file", "load_timestamp"))
        w = Window.partitionBy("record_key").orderBy(F.col("source_file").desc(),
                                                      F.col("source_payload_hash").desc())
        ready = ready.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")
        inserted, updated = merge_silver(ready, NOAA_SILVER)
        audit("noaa", started, "SUCCESS", total, inserted, updated, bad_n)
        print(f"NOAA SUCCESS: read {total}, inserted {inserted}, updated {updated}, quarantined {bad_n}")
        display(spark.table(NOAA_SILVER).limit(10))
    except Exception as e:
        audit("noaa", started, "FAILURE", total, bad=bad_n, error=str(e)[:1500])
        raise

## Cell 6 — QA: counts, duplicate keys, and operational logs

In [0]:
for table in [IBM_SILVER, NOAA_SILVER]:
    if spark.catalog.tableExists(table):
        print("\nTABLE:", table, "| Rows:", spark.table(table).count())
        duplicates = (spark.table(table).groupBy("record_key").count()
                      .filter(F.col("count") > 1).count())
        print("Duplicate business keys:", duplicates)
        assert duplicates == 0, f"Duplicates found in {table}"

print("\nMOST RECENT PIPELINE RUNS")
display(spark.table(LOG_TABLE).orderBy(F.col("started_at").desc()).limit(20))
print("\nQUARANTINE SUMMARY")
display(spark.table(BAD_TABLE).groupBy("dataset", "error_reason").count())


TABLE: workspace.silver.ibm_qubit_calibration | Rows: 456612
Duplicate business keys: 0

TABLE: workspace.silver.noaa_environment | Rows: 2194
Duplicate business keys: 0

MOST RECENT PIPELINE RUNS


run_id,stage,dataset,mode,parameters,started_at,finished_at,status,input_rows,rows_inserted,rows_updated,rows_quarantined,error_message,load_timestamp
bcc8b5ec-26d2-4698-b5b8-1205798e552d,Bronze-to-Silver,ibm,incremental,source_file=ibm_incremental_20261009T124928Z.csv; start_date=ALL; end_date=ALL,2026-10-10T14:09:39.707Z,2026-10-10T14:10:01.147Z,SUCCESS,33228,0,0,0,null,2026-10-10T14:10:01.147Z
f3068429-95e6-428f-867b-40ffdebad81f,Bronze-to-Silver,noaa,full,source_file=ALL; start_date=ALL; end_date=ALL,2026-10-10T11:07:01.117Z,2026-10-10T11:07:28.426Z,SUCCESS,2194,0,0,0,null,2026-10-10T11:07:28.426Z
f3068429-95e6-428f-867b-40ffdebad81f,Bronze-to-Silver,ibm,full,source_file=ALL; start_date=ALL; end_date=ALL,2026-10-10T11:06:19.356Z,2026-10-10T11:06:51.186Z,SUCCESS,456612,0,0,0,null,2026-10-10T11:06:51.186Z
325fe81a-e077-46dc-818c-103927a4b128,Bronze-to-Silver,ibm,backfill,source_file=ALL; start_date=2026-09-25; end_date=2026-09-26,2026-10-10T10:00:57.982Z,2026-10-10T10:01:07.370Z,SUCCESS,4680,0,0,0,null,2026-10-10T10:01:07.370Z
2e9033c9-6fce-4009-aad6-0ec0d8641382,Bronze-to-Silver,ibm,backfill,source_file=ALL; start_date=2026-09-25; end_date=2026-09-26,2026-10-10T09:55:27.811Z,2026-10-10T09:55:37.912Z,SUCCESS,4680,0,0,0,null,2026-10-10T09:55:37.912Z
45805151-0458-43e6-9d70-d7b8cd8d5607,Bronze-to-Silver,noaa,full,source_file=ALL; start_date=ALL; end_date=ALL,2026-10-10T09:09:14.804Z,2026-10-10T09:09:33.978Z,SUCCESS,2194,0,0,0,null,2026-10-10T09:09:33.978Z
45805151-0458-43e6-9d70-d7b8cd8d5607,Bronze-to-Silver,ibm,full,source_file=ALL; start_date=ALL; end_date=ALL,2026-10-10T09:08:53.409Z,2026-10-10T09:09:11.013Z,SUCCESS,456612,0,0,0,null,2026-10-10T09:09:11.013Z
4fca078b-f2f5-4157-a7fd-114de98928b5,Bronze-to-Silver,noaa,full,source_file=ALL; start_date=ALL; end_date=ALL,2026-10-10T08:52:05.833Z,2026-10-10T08:52:33.983Z,SUCCESS,2194,2194,0,0,null,2026-10-10T08:52:33.983Z
4fca078b-f2f5-4157-a7fd-114de98928b5,Bronze-to-Silver,ibm,full,source_file=ALL; start_date=ALL; end_date=ALL,2026-10-10T08:34:32.741Z,2026-10-10T08:35:05.551Z,SUCCESS,456612,456612,0,0,null,2026-10-10T08:35:05.551Z



QUARANTINE SUMMARY


dataset,error_reason,count


## How to demonstrate this notebook to your teacher

1. **Full:** `dataset=both`, `mode=full`, other widgets empty; run all cells. Take a screenshot of both Silver tables and the logs.
2. **Idempotency:** run all again with exactly the same inputs; the Silver row counts stay the same and MERGE should show `inserted=0, updated=0` (assuming Bronze has not changed).
3. **Incremental:** set `mode=incremental`, put one *real* Bronze `source_file` name in the widget, and set `dataset=ibm` or `noaa` accordingly. Run again. A new batch adds only new keys.
4. **Backfill:** set `mode=backfill`, `source_file` blank, and choose `start_date=2026-09-01`, `end_date=2026-09-07`. Run again. Historical rows can be replayed without duplicate keys.
5. **Bad data/schema drift:** the code places invalid numeric values or unexpected JSON fields in `workspace.silver.quarantined_records`; successful rows continue into Silver. Ask your friend for a safe *test record in a dedicated test Bronze table* if the teacher requires a live fault-injection demonstration—do not corrupt production Bronze.

**Note:** Every execution creates new **audit log** rows. That is intentional. Idempotency means the *Silver business data* remains unchanged on identical reruns, not that audit history stops growing.

Quarantine Record Check

In [0]:
test_bad = spark.createDataFrame(
    [("QA_TEST_ONLY.json", '{"unexpected_field":123}', "SCHEMA_DRIFT_NEW_COLUMN")],
    StructType([
        StructField("source_file", StringType(), True),
        StructField("raw_payload", StringType(), True),
        StructField("error_reason", StringType(), True)
    ])
)

quarantine(test_bad, "ibm")

display(
    spark.table(BAD_TABLE)
    .filter(F.col("source_file") == "QA_TEST_ONLY.json"))

record_key,dataset,source_file,raw_payload,error_reason,load_timestamp
e2e023a2a1c515e99d0bb664279f4078b569903a33b59c18e963c640fa858764,ibm,QA_TEST_ONLY.json,"{""unexpected_field"":123}",SCHEMA_DRIFT_NEW_COLUMN,2026-10-10T14:10:18.212Z


In [0]:
%sql
select * 
from workspace.silver.quarantined_records


record_key,dataset,source_file,raw_payload,error_reason,load_timestamp
e2e023a2a1c515e99d0bb664279f4078b569903a33b59c18e963c640fa858764,ibm,QA_TEST_ONLY.json,"{""unexpected_field"":123}",SCHEMA_DRIFT_NEW_COLUMN,2026-10-10T14:10:18.212Z


In [0]:
%sql
delete from workspace.silver.quarantined_records
where source_file= 'QA_TEST_ONLY.json'

num_affected_rows
1


In [0]:
%sql
select *
from workspace.silver.quarantined_records

record_key,dataset,source_file,raw_payload,error_reason,load_timestamp
